# Probabilistic N-gram Language Model for Darija

This notebook walks through the full pipeline of building and evaluating an interpolated trigram language model trained on a Darija (Moroccan Arabic) corpus.

**Approach:** Linear interpolation combining unigram, bigram, and trigram probabilities.
Lambda weights are estimated from training data via **deleted interpolation** (Jelinek & Mercer, 1980).

$$P(w \mid w_1, w_2) = \lambda_1 P(w) + \lambda_2 P(w \mid w_2) + \lambda_3 P(w \mid w_1, w_2)$$

## 1. Setup

In [ ]:
import random
from data_loader import load_corpus
from preprocessor import preprocess
from ngram_model import NgramModel
from evaluator import perplexity
from generator import generate

SEED = 42
random.seed(SEED)

## 2. Load Corpus

Training data: first 4 story files (~80 MB) + first 50 MB of Twitter data.

In [ ]:
texts = load_corpus()

## 3. Preprocessing

Steps applied:
- Diacritic (tashkeel) removal
- Arabic letter normalization (`أ إ آ` → `ا`, `ة` → `ه`, etc.)
- Noise removal: URLs, mentions, hashtags
- Franco-Arabe digits preserved (3=ع, 7=ح, 9=ق)
- Sentence segmentation + `<s>` / `</s>` boundary tokens

In [ ]:
sentences = preprocess(texts)
print(f"Total sentences : {len(sentences):,}")
print(f"Example sentence: {sentences[0]}")

In [ ]:
def split(sentences, train_ratio=0.8, val_ratio=0.1):
    random.shuffle(sentences)
    n = len(sentences)
    t = int(n * train_ratio)
    v = int(n * (train_ratio + val_ratio))
    return sentences[:t], sentences[t:v], sentences[v:]

train, val, test = split(sentences)
print(f"Train : {len(train):,}")
print(f"Val   : {len(val):,}")
print(f"Test  : {len(test):,}")

## 4. Training

Words appearing fewer than 5 times are replaced with `<UNK>` to reduce vocabulary sparsity.
Both a **bigram** and **trigram** model are trained for comparison.

In [ ]:
print("Training bigram model...")
bigram_model = NgramModel(order=2)
bigram_model.train(train)

In [ ]:
print("Training trigram model...")
trigram_model = NgramModel(order=3)
trigram_model.train(train)

## 5. Evaluation — Perplexity

$$PP = 2^{-\frac{1}{N} \sum \log_2 P(w_i \mid w_{i-2}, w_{i-1})}$$

Lower perplexity = better model fit.

In [ ]:
results = {}
for name, model in [("Bigram", bigram_model), ("Trigram", trigram_model)]:
    results[name] = {
        "Train":      perplexity(model, train[:5_000]),
        "Validation": perplexity(model, val),
        "Test":       perplexity(model, test),
    }

print(f"{'Model':<10} {'Train':>12} {'Validation':>12} {'Test':>12}")
print("-" * 48)
for name, scores in results.items():
    print(f"{name:<10} {scores['Train']:>12,.2f} {scores['Validation']:>12,.2f} {scores['Test']:>12,.2f}")

**Interpretation:**
- The trigram model achieves lower perplexity than bigram on all splits, confirming the value of wider context.
- The train/validation gap is expected for word-level n-gram models on noisy social media text: the vocabulary is large relative to the corpus, so many contexts seen at test time were never observed during training. This is a known limitation of count-based models.

## 6. Text Generation

Two strategies:
- **Sampling**: draws the next word proportionally to its probability
- **Greedy (top-5)**: samples from the 5 highest-probability words to avoid deterministic loops

In [ ]:
print("=== Sampling ===")
for i in range(5):
    print(f"  {i+1}. {generate(trigram_model, strategy='sample')}")

In [ ]:
print("=== Greedy (top-5) ===")
for i in range(3):
    print(f"  {i+1}. {generate(trigram_model, strategy='greedy')}")

In [ ]:
print("=== Seeded generation ===")
for seed in ["واش نتا", "كنت نحب", "الله يعطيك"]:
    print(f"  [{seed}] → {generate(trigram_model, seed=seed, strategy='sample')}")

## 7. Save Model

In [ ]:
trigram_model.save("ngram_model.pkl")
print("Trigram model saved.")